# Global Partner Demand Forecaster

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

## Load Trade Time-Series

In [ ]:
data_paths = [
    'data/final_csv/01_partner_discovery_india_as_exporter_eda.csv',
    '../data/final_csv/01_partner_discovery_india_as_exporter_eda.csv',
    'backend/brain/data/final_csv/01_partner_discovery_india_as_exporter_eda.csv',
    'brain/datasets/dump/brain_prev/data_pipeline/data/final_csv/01_partner_discovery_india_as_exporter_eda.csv'
]
path = next(p for p in data_paths if os.path.exists(p))
df = pd.read_csv(path)
df.head()

## Demand Trends & Lags

In [ ]:
val_col = 'trade_value_usd' if 'trade_value_usd' in df.columns else 'net_weight_kg'
if 'year' in df.columns and 'partner_iso3' in df.columns:
    df = df.sort_values(['partner_iso3', 'year'])
    df['ma3'] = df.groupby('partner_iso3')[val_col].transform(lambda s: s.rolling(3, min_periods=1).mean())
    df['yoy_growth'] = df.groupby('partner_iso3')[val_col].pct_change().fillna(0)

sample_partner = df['partner_iso3'].value_counts().index[0]
sample_df = df[df['partner_iso3'] == sample_partner]
plt.figure(figsize=(10, 4))
plt.plot(sample_df['year'], sample_df[val_col], marker='o', label='Actual')
if 'ma3' in sample_df:
    plt.plot(sample_df['year'], sample_df['ma3'], linestyle='--', label='MA3 Baseline')
plt.title(f"Demand Trajectory: {sample_partner}")
plt.legend()
plt.show()

## XGBoost Quantile Regressors (Q10, Q50, Q90)

In [ ]:
features = ['ma3', 'yoy_growth'] if 'ma3' in df.columns else [val_col]
X = df[features].fillna(0).values
y = df[val_col].fillna(0).values

quantiles = [0.1, 0.5, 0.9]
models = {}
for q in quantiles:
    m = xgb.XGBRegressor(objective='reg:quantileerror', quantile_alpha=q, n_estimators=50, max_depth=3)
    m.fit(X, y)
    models[q] = m

preds = {q: models[q].predict(X) for q in quantiles}

## Forecast & 80% Prediction Band

In [ ]:
res = pd.DataFrame({
    'actual': y[:10],
    'q10_lower': np.round(preds[0.1][:10], 1),
    'q50_point': np.round(preds[0.5][:10], 1),
    'q90_upper': np.round(preds[0.9][:10], 1)
})
res